<div style="display: flex; align-items: center;">
  <img src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/DecisionIntelligenceWorkshopLogo.png" width="40px" style="margin-right: 10px;">
  <span style="font-size: 1.5em; font-weight: bold;">Workshop (AI Extensions) - Optimizing with Decision Frames</span>
</div>

Decision Intelligence applied in this module:

* Using decision frames to evaluate a major purchase from multiple perspectives.
* Applying Edward de Bono's Six Thinking Hats as reusable decision lenses.
* Asking GenAI to prioritize the most useful frames for a high-stakes decision.
* Combining prompt design and Microsoft.Extensions.AI to produce strict Markdown-table outputs.

### Step 1 - Initialize Configuration Builder & Build the AI Extensions Responses API Orchestration

In [ ]:
#r "nuget: Microsoft.Extensions.Configuration, 10.0.8"
#r "nuget: Microsoft.Extensions.Configuration.Json, 10.0.8"
#r "nuget: System.Text.Json, 10.0.8"

using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using System.IO;
using System;

var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// IMPORTANT: You ONLY NEED either Azure OpenAI or OpenAI connection info, not both.
// Azure OpenAI Connection Info
var azureOpenAIEndpoint = config["AzureOpenAI:Endpoint"];
var azureOpenAIAPIKey = config["AzureOpenAI:APIKey"];
var azureOpenAIModelDeploymentName = config["AzureOpenAI:ModelDeploymentName"];
// OpenAI Connection Info 
var openAIAPIKey = config["OpenAI:APIKey"];
var openAIModelId = config["OpenAI:ModelId"];

In [ ]:
// Import the Microsoft Extensions AI NuGet Packages
#r "nuget: Microsoft.Extensions.AI, 10.6.0"
#r "nuget: Microsoft.Extensions.AI.Abstractions, 10.6.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.6.0"
// Import Azure & OpenAI NuGet Packages
#r "nuget: Azure.AI.OpenAI, 2.9.0-beta.1"
#r "nuget: Azure.Identity, 1.21.0"
#r "nuget: OpenAI, 2.10.0"

using Azure.AI.OpenAI;
using Microsoft.Extensions.AI;
using OpenAI;
using OpenAI.Responses;
using System.ClientModel;
using System.Collections.Generic;
using System.Threading.Tasks;

// Set the flag to use Azure OpenAI or OpenAI. False to use OpenAI, True to use Azure OpenAI.
var useAzureOpenAI = true;

// Create the IChatClient based on the selected service.
IChatClient chatClient;

#pragma warning disable OPENAI001

if (useAzureOpenAI)
{
    Console.WriteLine("Using Azure OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey);
    var azureOpenAIClient = new AzureOpenAIClient(new Uri(azureOpenAIEndpoint), apiKeyCredential);

    // Get the ResponsesClient from AzureOpenAIClient and adapt it to Microsoft.Extensions.AI.
    var responsesClient = azureOpenAIClient.GetResponsesClient();
    chatClient = responsesClient.AsIChatClient(defaultModelId: azureOpenAIModelDeploymentName);
}
else
{
    Console.WriteLine("Using OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(openAIAPIKey);
    var openAIClient = new OpenAIClient(apiKeyCredential);

    // Get the ResponsesClient from OpenAIClient and adapt it to Microsoft.Extensions.AI.
    var responsesClient = openAIClient.GetResponsesClient();
    chatClient = responsesClient.AsIChatClient(defaultModelId: openAIModelId);
}

#pragma warning restore OPENAI001

var decisionFrameSystemPrompt = """
You are a Decision Intelligence assistant.
Help the user explore options, evaluate tradeoffs, reason through uncertainty, solve problems,
and apply systems thinking to personal, professional, strategic, and operational decisions.

Provide responses that are structured, logical, balanced, analytical, and pragmatic.
Improve the user's judgment rather than simply making the choice for them.

Output Format Instructions:
Return only a valid Markdown table.
Do not include prose before or after the table.
Do not wrap the table in triple backticks.
Do not use Markdown headings anywhere in the response.
Do not use #, ##, ###, ####, or any heading syntax inside or outside table cells.
Do not use horizontal rules of any kind.
Do not use ---, ***, or ___ anywhere in the response.
Do not use bullet lists or numbered lists inside table cells.
Use plain text column headers only.
If line breaks are needed inside a cell, use <br>.
Escape any literal pipe characters inside cell content so the table structure is not broken.
Make the table syntactically valid Markdown with exactly one header row and one separator row.
""";

async Task<string> RunDecisionFramePromptAsync(string userPrompt, ChatOptions? options = null)
{
    var chatMessages = new List<ChatMessage>
    {
        new(ChatRole.System, decisionFrameSystemPrompt),
        new(ChatRole.User, userPrompt)
    };

    ChatResponse response = await chatClient.GetResponseAsync(chatMessages, options);
    return response.Text ?? string.Empty;
}

#pragma warning disable OPENAI001

var decisionFrameChatOptions = new ChatOptions
{
    RawRepresentationFactory = _ => new OpenAI.Responses.CreateResponseOptions
    {
        Model = useAzureOpenAI ? azureOpenAIModelDeploymentName : openAIModelId,
        ReasoningOptions = new OpenAI.Responses.ResponseReasoningOptions
        {
            ReasoningEffortLevel = OpenAI.Responses.ResponseReasoningEffortLevel.Medium,
            ReasoningSummaryVerbosity = OpenAI.Responses.ResponseReasoningSummaryVerbosity.Detailed
        },
        StoredOutputEnabled = false
    }
};

#pragma warning restore OPENAI001

### Step 2 - Decision Frames (Thinking Hats) Scenario for a Major Purchase

> 📜 **_"The mere formulation of a problem is far more essential than its solution, which may be merely a matter of mathematical or experimental skill. To raise new questions, new possibilities, to regard old problems from a new angle requires creative imagination and marks real advances in science."_**
>
> -- <cite>Albert Einstein (Theoretical Physicist, Best known for developing the theory of relativity)</cite>  

The code below lists potential decision frames using the Six Thinking Hats framework for purchasing a house. Generative AI can help apply each hat as a distinct decision lens, making it easier to inspect the same choice from multiple useful perspectives.

In [ ]:
// Decision Frames Prompt that lists the Six Thinking Hats approach to purchasing a house.
var decisionFramesPrompt = """
You are planning on purchasing a new home.
Use Edward de Bono's Six Thinking Hats to frame the decision-making process.

Instructions:
Create one row for each of the six thinking hats.
For each row, explain how that decision frame should guide the house-purchase decision.
Include practical questions and evidence the buyer should gather.

Output Requirements:
Return one Markdown table with these columns:
Decision Frame | Focus Area | House-Purchase Questions | Evidence To Gather | Decision Value
""";

var decisionFramesResponseText = await RunDecisionFramePromptAsync(
    decisionFramesPrompt,
    decisionFrameChatOptions);

decisionFramesResponseText.DisplayAs("text/markdown");

This can be enhanced further by asking GenAI to suggest which decision frames might be most useful for a high-stakes decision. The prompt below lists all of the decision frames, prioritizes them by expected impact and influence, and recommends which frames deserve the most attention.

This lets the AI automation help optimize the decision process while the human decision maker remains responsible for the actual decision selection and execution.

In [ ]:
// Decision Frames Prompt that ranks the Six Thinking Hats approach to purchasing a house.
var decisionFramesListPrompt = """
You are planning on purchasing a new home.
Use Edward de Bono's Six Thinking Hats to frame the decision-making process.

Instructions:
List the six thinking hats in the order of expected impact and influence from most to least for a house-purchase decision.
For each thinking hat, explain why it has that ranking.
Recommend whether the buyer should focus on that hat heavily, moderately, or lightly.

Output Requirements:
Return one Markdown table with these columns:
Rank | Decision Frame | Why It Matters For A House Purchase | Impact / Influence | Recommended Focus
""";

var decisionFramesListResponseText = await RunDecisionFramePromptAsync(
    decisionFramesListPrompt,
    decisionFrameChatOptions);

decisionFramesListResponseText.DisplayAs("text/markdown");

So far, GenAI has helped create decision frames with the Six Thinking Hats framework, rank them from most to least impactful, and identify where to focus attention. The next step is to apply the selected decision frames directly to the house-purchase decision.

This notebook uses a single optimized prompt for that final step: the model performs the ranking and frame selection internally, then returns only the applied decision-frame approach as a Markdown table.

#### Single Decision Prompt

In [ ]:
// Decision Frames Prompt that internally ranks and selects the strongest frames, then only returns the applied approach.
var optimizedDecisionFramesPrompt = """
You are planning on purchasing a new home.
Use Edward de Bono's Six Thinking Hats to frame the decision-making process.

Internal Instructions:
Internally identify an approach for each of the six thinking hats.
Internally rank the thinking hats by expected impact and influence for the house-purchase decision.
Internally select the most useful decision frames to focus on.
Do not show the internal ranking, hidden selection notes, or hidden reasoning steps.

Visible Output Instructions:
From the internally selected decision frames, create an applied approach for purchasing a house.
The response should help the buyer know what to inspect, what questions to ask, what evidence to gather, and what action to take.

Output Requirements:
Return one Markdown table with these columns:
Selected Decision Frame | Applied House-Purchase Approach | Key Questions | Evidence / Signals | Buyer Action
""";

var optimizedDecisionFramesResponseText = await RunDecisionFramePromptAsync(
    optimizedDecisionFramesPrompt,
    decisionFrameChatOptions);

optimizedDecisionFramesResponseText.DisplayAs("text/markdown");